# 05_testing — Three patterns for testing LangGraph controllers

Testing a `LangGraphController` doesn't require a running machine.
This notebook demonstrates three standalone test patterns.

**What's new (vs 04):**

| Pattern | Description |
|---|---|
| **1 — Structural** | Call `.build()`, assert no exception; tests topology + data-contract |
| **2 — Stub Action** | Replace real Actions with fixed-result stubs; isolates routing logic |
| **3 — Mock box** | Call `ctrl.ainvoke()` with a `MagicMock` box; tests full ainvoke path |

In [ ]:
!pip install -q "aoa-action-machine" "langgraph"

In [ ]:
from unittest.mock import AsyncMock, MagicMock

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.langgraph import LangGraphController

## Step 1 — Domain, Params, Results

In [ ]:
class SupportDomain(BaseDomain):
    name = "support"
    description = "Support ticket processing"

class ClassifyParams(BaseParams):
    ticket_id: str = Field(description="Ticket identifier")
    note: str = Field(default="", description="Ticket content")

class ClassifyResult(BaseResult):
    category: str = Field(description="Ticket category: bug | feature | billing")

class ResolveParams(BaseParams):
    ticket_id: str = Field(description="Ticket identifier")
    category: str = Field(description="Ticket category")

class ResolveResult(BaseResult):
    resolved: bool = Field(description="Resolution flag")
    resolution_note: str = Field(description="Resolution note")

## Step 2 — Real Actions and Stub Actions

Stubs use the same Params/Result types as real Actions so topology validation still passes.
The `@summary_aspect` returns a fixed result without real logic.

In [ ]:
@meta(description="Classify ticket by content", domain=SupportDomain)
@check_roles(GuestRole)
class ClassifyTicketAction(BaseAction[ClassifyParams, ClassifyResult]):
    @summary_aspect("Classify ticket")
    async def classify_summary(self, params, state, box, connections):
        text = params.note.lower()
        if "billing" in text or "invoice" in text or "payment" in text:
            cat = "billing"
        elif "crash" in text or "error" in text or "bug" in text:
            cat = "bug"
        else:
            cat = "feature"
        return ClassifyResult(category=cat)

@meta(description="Route ticket to engineering", domain=SupportDomain)
@check_roles(GuestRole)
class EngineeringAction(BaseAction[ResolveParams, ResolveResult]):
    @summary_aspect("Assign to engineering")
    async def resolve_summary(self, params, state, box, connections):
        tag = params.category.upper()
        return ResolveResult(
            resolved=True,
            resolution_note=f"[{tag}] #{params.ticket_id} → engineering",
        )

@meta(description="Route ticket to billing team", domain=SupportDomain)
@check_roles(GuestRole)
class BillingAction(BaseAction[ResolveParams, ResolveResult]):
    @summary_aspect("Assign to billing")
    async def resolve_summary(self, params, state, box, connections):
        return ResolveResult(
            resolved=True,
            resolution_note=f"[BILLING] #{params.ticket_id} → billing team",
        )

# Stubs — hard-coded results, same Params/Result contracts
@meta(description="[Stub] Always classifies as bug", domain=SupportDomain)
@check_roles(GuestRole)
class StubClassifyAction(BaseAction[ClassifyParams, ClassifyResult]):
    @summary_aspect("Stub: classify as bug")
    async def classify_summary(self, params, state, box, connections):
        return ClassifyResult(category="bug")

@meta(description="[Stub] Always resolves", domain=SupportDomain)
@check_roles(GuestRole)
class StubEngineeringAction(BaseAction[ResolveParams, ResolveResult]):
    @summary_aspect("Stub: always resolved")
    async def resolve_summary(self, params, state, box, connections):
        return ResolveResult(resolved=True, resolution_note="[STUB] resolved")

## Step 3 — Graph builder helper

Accepts pluggable Action classes so the same topology can be tested with real or stub Actions.

In [ ]:
def _build_graph(
    classify_cls: type = ClassifyTicketAction,
    engineering_cls: type = EngineeringAction,
) -> LangGraphController:
    return (
        LangGraphController()
        .inp("ticket_id", str, "Ticket identifier")
        .inp("note", str, "Ticket content")
        .mid("category", str, "Ticket category: bug | feature | billing")
        .mid("resolved", bool, "Resolution flag")
        .mid("resolution_note", str, "Resolution note")
        .out("category")
        .out("resolved")
        .out("resolution_note")
        .node(classify_cls)
        .node(engineering_cls)
        .node(BillingAction)
        .start(classify_cls)
        .route(
            classify_cls,
            on=lambda s: s.category,
            paths={
                "bug": engineering_cls,
                "feature": engineering_cls,
                "billing": BillingAction,
            },
        )
        .finish(engineering_cls)
        .finish(BillingAction)
        .build()
    )

## Pattern 1 — Structural test

`.build()` validates topology and data-contract statically. No machine, no box, no network.

In [ ]:
ctrl = _build_graph()
assert ctrl._built is True
print("Pattern 1 ✓  build() validates topology successfully")

## Pattern 2 — Stub Actions

Replace real Actions with stubs that always classify as `"bug"` and always resolve.
Useful for testing routing logic in isolation.

In [ ]:
ctrl_stub = _build_graph(
    classify_cls=StubClassifyAction,
    engineering_cls=StubEngineeringAction,
)
mock_result = MagicMock()
mock_result.model_dump.return_value = {"resolved": True, "resolution_note": "[STUB] resolved"}
box = MagicMock()
box.run = AsyncMock(return_value=mock_result)

result = await ctrl_stub.ainvoke({"ticket_id": "T-9", "note": "anything"}, box)

assert result["category"] == "bug"   # StubClassifyAction always → "bug"
assert result["resolved"] is True
print("Pattern 2 ✓  stub Actions route correctly")

## Pattern 3 — Mock box

Use a custom `box.run` function to simulate Action execution and assert the full ainvoke path.

In [ ]:
ctrl_real = _build_graph()

classify_result = MagicMock()
classify_result.model_dump.return_value = {"category": "billing"}

billing_result = MagicMock()
billing_result.model_dump.return_value = {
    "resolved": True,
    "resolution_note": "[BILLING] #T-5 → billing team",
}

call_count = 0

async def fake_run(action_cls, params, connections=None):
    global call_count
    call_count += 1
    if action_cls is ClassifyTicketAction:
        return classify_result
    return billing_result

box3 = MagicMock()
box3.run = fake_run

result3 = await ctrl_real.ainvoke({"ticket_id": "T-5", "note": "invoice issue"}, box3)

assert result3["category"] == "billing"
assert result3["resolved"] is True
assert call_count == 2  # classify + billing
print(f"Pattern 3 ✓  mock box captured {call_count} Action calls, output: {result3['category']}")